In [14]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from keras.callbacks import EarlyStopping, LearningRateScheduler

In [15]:
# Upload the padding data

X_train_padded = np.load(
    "../Dataset/processed/X_train_padded.npy"
)

X_val_padded = np.load(
    "../Dataset/processed/X_val_padded.npy"
)

X_test_padded = np.load(
    "../Dataset/processed/X_test_padded.npy"
)

y_train = np.load(
    "../Dataset/processed/y_train.npy"
)

y_val = np.load(
    "../Dataset/processed/y_val.npy"
)

y_test = np.load(
    "../Dataset/processed/y_test.npy"
)

In [16]:
with open(
    "../Dataset/processed/feature_config.pkl",
    "rb"
) as file:

    feature_config = pickle.load(file)

In [17]:

VOCAB_SIZE = feature_config["vocab_size"]
EMBEDDING_DIM = feature_config["embedding_dim"]
MAX_SEQUENCE_LENGTH = feature_config["max_sequence_length"]

print("Vocabulary size:", VOCAB_SIZE)
print("Embedding dimension:", EMBEDDING_DIM)
print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)

Vocabulary size: 20000
Embedding dimension: 128
Maximum sequence length: 200


In [18]:
# Load TF-IDF Features for ANN

with open(
    "../Dataset/processed/tfidf_vectorizer.pkl",
    "rb"
) as file:

    tfidf_vectorizer = pickle.load(file)

In [19]:
with open(
    "../Dataset/processed/text_splits.pkl",
    "rb"
) as file:

    text_splits = pickle.load(file)

In [20]:
X_train_text = text_splits["X_train"]
X_val_text = text_splits["X_val"]
X_test_text = text_splits["X_test"]

In [21]:
# Fit already fitted TF-IDF vectorizer

X_train_tfidf = tfidf_vectorizer.transform(
    X_train_text
)

X_val_tfidf = tfidf_vectorizer.transform(
    X_val_text
)

X_test_tfidf = tfidf_vectorizer.transform(
    X_test_text
)

In [22]:
print("TF-IDF train:", X_train_tfidf.shape)
print("TF-IDF validation:", X_val_tfidf.shape)
print("TF-IDF test:", X_test_tfidf.shape)

TF-IDF train: (34705, 20000)
TF-IDF validation: (7439, 20000)
TF-IDF test: (7438, 20000)


# ANN

In [23]:
TFIDF_FEATURES = X_train_tfidf.shape[1]

In [25]:
# Dropout

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),
    
    
    Dense(
        1,
        activation="sigmoid"
    )
])

E0000 00:00:1787561036.230209  215937 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [26]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [27]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

W0000 00:00:1787561069.034392  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.8692 - loss: 0.3333 - val_accuracy: 0.9032 - val_loss: 0.2430
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 39ms/step - accuracy: 0.9457 - loss: 0.1507 - val_accuracy: 0.8961 - val_loss: 0.2684
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.9759 - loss: 0.0759 - val_accuracy: 0.8896 - val_loss: 0.3467
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9891 - loss: 0.0374 - val_accuracy: 0.8919 - val_loss: 0.4082
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - accuracy: 0.9956 - loss: 0.0165 - val_accuracy: 0.8892 - val_loss: 0.5153
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 22s 41ms/step - accuracy: 0.9971 - loss: 0.0096 - val_accuracy: 0.8907 - val_loss: 0.5869
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9981 - loss: 0.0061 - val_accuracy: 0.8887 - val_loss: 0.6611
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9985 - loss: 0.0040 - 

In [28]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step


In [29]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [30]:
def evaluate_model(y_true, y_pred, y_prob):
    
    accuracy = accuracy_score(
        y_true,
        y_pred
    )
    
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )
    
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )
    
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )
    
    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )
    
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

In [31]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [32]:
ann_metrics

{'Accuracy': 0.8827641839204087,
 'Precision': 0.9001398601398601,
 'Recall': 0.8620412536833646,
 'F1 Score': 0.8806787082649151,
 'ROC-AUC': 0.953827102116188}

In [33]:
# Dropout 0.5

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [34]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [35]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

W0000 00:00:1787561654.203858  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.8525 - loss: 0.3633 - val_accuracy: 0.9042 - val_loss: 0.2405
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 37s 38ms/step - accuracy: 0.9380 - loss: 0.1766 - val_accuracy: 0.9040 - val_loss: 0.2510
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 37s 31ms/step - accuracy: 0.9634 - loss: 0.1124 - val_accuracy: 0.8986 - val_loss: 0.2899
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9775 - loss: 0.0718 - val_accuracy: 0.8990 - val_loss: 0.3430
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9858 - loss: 0.0452 - val_accuracy: 0.8958 - val_loss: 0.4001
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - accuracy: 0.9901 - loss: 0.0316 - val_accuracy: 0.8949 - val_loss: 0.4313
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - accuracy: 0.9933 - loss: 0.0213 - val_accuracy: 0.8949 - val_loss: 0.4992
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - accuracy: 0.9941 - loss: 0.0182 - 

In [36]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step


In [37]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [38]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [39]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9551873667147117}

In [41]:
# Batchnormalization

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense( 64 ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),
    
    tf.keras.layers.Dropout(0.5),

    Dense(32),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [42]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [43]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

W0000 00:00:1787562669.683050  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - accuracy: 0.8286 - loss: 0.3829 - val_accuracy: 0.8997 - val_loss: 0.2743
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9234 - loss: 0.2066 - val_accuracy: 0.8968 - val_loss: 0.2568
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9521 - loss: 0.1362 - val_accuracy: 0.8973 - val_loss: 0.3021
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9665 - loss: 0.0960 - val_accuracy: 0.8939 - val_loss: 0.3434
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9742 - loss: 0.0730 - val_accuracy: 0.8934 - val_loss: 0.3761
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9777 - loss: 0.0638 - val_accuracy: 0.8914 - val_loss: 0.3860
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - accuracy: 0.9820 - loss: 0.0534 - val_accuracy: 0.8903 - val_loss: 0.4145
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9851 - loss: 0.0449 - 

In [44]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [45]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [46]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9544096801586897}

In [47]:
# BatchNormalization without dropout

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense( 64 ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    Dense(32),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [48]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [49]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

W0000 00:00:1787562950.233311  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.8707 - loss: 0.3048 - val_accuracy: 0.8931 - val_loss: 0.2656
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9687 - loss: 0.0897 - val_accuracy: 0.8868 - val_loss: 0.3196
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 32ms/step - accuracy: 0.9897 - loss: 0.0338 - val_accuracy: 0.8773 - val_loss: 0.4074
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.9927 - loss: 0.0229 - val_accuracy: 0.8851 - val_loss: 0.4526
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9939 - loss: 0.0190 - val_accuracy: 0.8822 - val_loss: 0.4552
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - accuracy: 0.9959 - loss: 0.0123 - val_accuracy: 0.8814 - val_loss: 0.5147
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - accuracy: 0.9972 - loss: 0.0097 - val_accuracy: 0.8816 - val_loss: 0.5481
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9976 - loss: 0.0080 - 

In [50]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [51]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [52]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9476206847560493}

In [53]:
# New Optimizers

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [54]:
ann_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [55]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

W0000 00:00:1787563444.736503  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.8327 - loss: 0.4069 - val_accuracy: 0.9021 - val_loss: 0.2422
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9213 - loss: 0.2188 - val_accuracy: 0.9079 - val_loss: 0.2383
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9398 - loss: 0.1762 - val_accuracy: 0.9052 - val_loss: 0.2521
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9523 - loss: 0.1468 - val_accuracy: 0.9039 - val_loss: 0.2762
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9599 - loss: 0.1265 - val_accuracy: 0.9004 - val_loss: 0.2982
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9667 - loss: 0.1112 - val_accuracy: 0.8977 - val_loss: 0.3319
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9726 - loss: 0.0970 - val_accuracy: 0.8946 - val_loss: 0.3599
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9751 - loss: 0.0869 - 

In [56]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [57]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [58]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9543755533406866}

In [ ]:
# Learning Rate

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activaation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [63]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.6187 - loss: 0.6803 - val_accuracy: 0.7816 - val_loss: 0.6496
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.7913 - loss: 0.6015 - val_accuracy: 0.8693 - val_loss: 0.5235
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.8464 - loss: 0.4717 - val_accuracy: 0.8868 - val_loss: 0.3851
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.8751 - loss: 0.3616 - val_accuracy: 0.8950 - val_loss: 0.3023
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.8909 - loss: 0.3007 - val_accuracy: 0.9004 - val_loss: 0.2649
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9023 - loss: 0.2675 - val_accuracy: 0.9042 - val_loss: 0.2480
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.9130 - loss: 0.2403 - val_accuracy: 0.9050 - val_loss: 0.2395
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9191 - loss: 0.2247 - 

In [64]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [65]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [66]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [67]:
ann_metrics

{'Accuracy': 0.9046786770637268,
 'Precision': 0.9044943820224719,
 'Recall': 0.9057058665952317,
 'F1 Score': 0.9050997189131308,
 'ROC-AUC': 0.9674303626733588}

In [68]:
ann_model.save(
    "../models/ann/ann_baseline_5_90.keras"
)

In [69]:
with open(
    "../models/ann/ann_history_5_90.pkl",
    "wb"
) as file:

    pickle.dump(
        ann_history.history,
        file
    )

In [70]:
# Adam 

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [71]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.9321 - loss: 0.1900 - val_accuracy: 0.9103 - val_loss: 0.2321
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9428 - loss: 0.1699 - val_accuracy: 0.9097 - val_loss: 0.2332
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9514 - loss: 0.1503 - val_accuracy: 0.9099 - val_loss: 0.2371
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9591 - loss: 0.1338 - val_accuracy: 0.9087 - val_loss: 0.2425
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9658 - loss: 0.1149 - val_accuracy: 0.9094 - val_loss: 0.2503
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9721 - loss: 0.0995 - val_accuracy: 0.9090 - val_loss: 0.2598
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9761 - loss: 0.0874 - val_accuracy: 0.9076 - val_loss: 0.2712
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.9817 - loss: 0.0738 - 

In [72]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [73]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [74]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [75]:
ann_metrics

{'Accuracy': 0.8962086582414628,
 'Precision': 0.8978769148078474,
 'Recall': 0.8949906241628717,
 'F1 Score': 0.8964314462033808,
 'ROC-AUC': 0.9634899443378584}

In [76]:
# Adam 

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.00025),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [77]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.9855 - loss: 0.0531 - val_accuracy: 0.9008 - val_loss: 0.3417
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9909 - loss: 0.0385 - val_accuracy: 0.9001 - val_loss: 0.3862
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9929 - loss: 0.0286 - val_accuracy: 0.8994 - val_loss: 0.4246
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9950 - loss: 0.0207 - val_accuracy: 0.8951 - val_loss: 0.4564
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9966 - loss: 0.0157 - val_accuracy: 0.8935 - val_loss: 0.5065
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9976 - loss: 0.0111 - val_accuracy: 0.8934 - val_loss: 0.5351
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9978 - loss: 0.0099 - val_accuracy: 0.8921 - val_loss: 0.5708
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9980 - loss: 0.0080 - 

In [78]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8881419736488303,
 'Precision': 0.8902340597255851,
 'Recall': 0.8864184302169836,
 'F1 Score': 0.8883221476510067,
 'ROC-AUC': 0.9565075756836299}

In [80]:
# Adam 

ann_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.00025),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [81]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.9991 - loss: 0.0042 - val_accuracy: 0.8942 - val_loss: 0.7272
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.9990 - loss: 0.0041 - val_accuracy: 0.8939 - val_loss: 0.7500
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9993 - loss: 0.0032 - val_accuracy: 0.8934 - val_loss: 0.7853
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9993 - loss: 0.0031 - val_accuracy: 0.8937 - val_loss: 0.7921
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9993 - loss: 0.0029 - val_accuracy: 0.8919 - val_loss: 0.8180
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9993 - loss: 0.0026 - val_accuracy: 0.8941 - val_loss: 0.8322
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9993 - loss: 0.0023 - val_accuracy: 0.8943 - val_loss: 0.8448
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9993 - loss: 0.0024 - 

In [82]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.8855875235278301,
 'Precision': 0.8824309978768577,
 'Recall': 0.8907045271899277,
 'F1 Score': 0.886548460205306,
 'ROC-AUC': 0.9553278144773626}

In [84]:
# Batch Size Tuning 

# Learning Rate

ann_model = Sequential([
    
    Input(shape=(TFIDF_FEATURES,)),
    
    Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [85]:
ann_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [86]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size= 32
)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - accuracy: 0.6724 - loss: 0.6701 - val_accuracy: 0.8646 - val_loss: 0.6063
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.8332 - loss: 0.5113 - val_accuracy: 0.8812 - val_loss: 0.3841
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.8724 - loss: 0.3500 - val_accuracy: 0.8931 - val_loss: 0.2821
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.8915 - loss: 0.2821 - val_accuracy: 0.9021 - val_loss: 0.2514
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.9066 - loss: 0.2502 - val_accuracy: 0.9054 - val_loss: 0.2409
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9142 - loss: 0.2315 - val_accuracy: 0.9064 - val_loss: 0.2362
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.9202 - loss: 0.2184 - val_accuracy: 0.9071 - val_loss: 0.2351
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9256 -

In [87]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.9049475665501479,
 'Precision': 0.9058476394849786,
 'Recall': 0.9046343423519957,
 'F1 Score': 0.9052405843720681,
 'ROC-AUC': 0.9672767919923446}

In [88]:
# Early Stopping 

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=20,
    batch_size= 64,

    callbacks=[
        early_stopping
    ]
)

Epoch 1/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9381 - loss: 0.1794 - val_accuracy: 0.9086 - val_loss: 0.2381
Epoch 2/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9402 - loss: 0.1753 - val_accuracy: 0.9078 - val_loss: 0.2389
Epoch 3/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9427 - loss: 0.1666 - val_accuracy: 0.9074 - val_loss: 0.2409


In [91]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.905216456036569,
 'Precision': 0.9007411328745368,
 'Recall': 0.9115992499330298,
 'F1 Score': 0.9061376647583544,
 'ROC-AUC': 0.9672536551665798}

In [92]:
# LR Schedule

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [93]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,
    
    validation_data=(
        X_val_tfidf,
        y_val
    ),
    
    epochs=10,
    batch_size= 64,

    callbacks=[
        lr_scheduler
    ]
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9396 - loss: 0.1746 - val_accuracy: 0.9075 - val_loss: 0.2395 - learning_rate: 1.0000e-04
Epoch 2/10
542/543 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9421 - loss: 0.1695
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9421 - loss: 0.1695 - val_accuracy: 0.9085 - val_loss: 0.2417 - learning_rate: 1.0000e-04
Epoch 3/10
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9451 - loss: 0.1647
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9451 - loss: 0.1647 - val_accuracy: 0.9074 - val_loss: 0.2421 - learning_rate: 5.0000e-05
Epoch 4/10
542/543 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9466 - loss: 0.1599
Epoch 4: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.94

In [94]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.9033342296316214,
 'Precision': 0.9016524520255863,
 'Recall': 0.9062416287168498,
 'F1 Score': 0.9039412157648631,
 'ROC-AUC': 0.9669734103645027}

In [ ]:
# RNN

